In [ ]:
# Clone the original repository to fetch the MNIST dataset files
!git clone https://github.com/mnielsen/neural-networks-and-deep-learning.git

In [ ]:
import pickle
import gzip
import numpy as np

# Defines a Python 3 compatible function to load data from the mnist.pkl.gz file
def load_data_wrapper_py3():
    with gzip.open('neural-networks-and-deep-learning/data/mnist.pkl.gz', 'rb') as f:
        training_data, validation_data, test_data = pickle.load(f, encoding='latin1')

    def train_inputs(d): return [np.reshape(x, (784, 1)) for x in d[0]]
    def train_results(j):
        e = np.zeros((10, 1))
        e[j] = 1.0
        return e

    training_inputs = train_inputs(training_data)
    training_results = [train_results(y) for y in training_data[1]]
    training_d = list(zip(training_inputs, training_results))

    validation_inputs = [np.reshape(x, (784, 1)) for x in validation_data[0]]
    validation_d = list(zip(validation_inputs, validation_data[1]))

    test_inputs = [np.reshape(x, (784, 1)) for x in test_data[0]]
    test_d = list(zip(test_inputs, test_data[1]))

    return training_d, validation_d, test_d

# Load the dataset using the wrapper function
training_data, validation_data, test_data = load_data_wrapper_py3()
print("SUCCESS: MNIST data loaded into memory for Python 3!")

In [ ]:
# CODE TO GENERATE THE ADVANCED NETWORK2.PY FILE FROM SCRATCH
network2_content = """
import json
import random
import sys
import numpy as np

class CrossEntropyCost(object):
    @staticmethod
    def fn(a, y):
        return np.sum(np.nan_to_num(-y*np.log(a)-(1-y)*np.log(1-a)))

    @staticmethod
    def delta(z, a, y):
        return (a-y)

class QuadraticCost(object):
    @staticmethod
    def fn(a, y):
        return 0.5*np.linalg.norm(a-y)**2

    @staticmethod
    def delta(z, a, y):
        return (a-y) * sigmoid_prime(z)

class Network(object):
    def __init__(self, sizes, cost=CrossEntropyCost):
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.default_weight_initializer()
        self.cost = cost

    def default_weight_initializer(self):
        self.biases = [np.random.randn(y, 1) for y in self.sizes[1:]]
        self.weights = [np.random.randn(y, x)/np.sqrt(x) for x, y in zip(self.sizes[:-1], self.sizes[1:])]

    def feedforward(self, a):
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a)+b)
        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta,
            lmbda = 0.0,
            evaluation_data=None,
            monitor_evaluation_accuracy=False):

        training_data = list(training_data)
        n = len(training_data)

        if evaluation_data:
            evaluation_data = list(evaluation_data)
            n_data = len(evaluation_data)

        for j in range(epochs):
            random.shuffle(training_data)
            mini_batches = [
                training_data[k:k+mini_batch_size]
                for k in range(0, n, mini_batch_size)]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta, lmbda, n)

            print(f"Epoch {j} training complete")
            if monitor_evaluation_accuracy and evaluation_data:
                accuracy = self.accuracy(evaluation_data)
                print(f"Accuracy on evaluation data: {accuracy} / {n_data}")
            print()

    def update_mini_batch(self, mini_batch, eta, lmbda, n):
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)
            nabla_b = [nb+dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw+dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]

        self.weights = [(1-eta*(lmbda/n))*w-(eta/len(mini_batch))*nw
                        for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b-(eta/len(mini_batch))*nb
                       for b, nb in zip(self.biases, nabla_b)]

    def backprop(self, x, y):
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        activation = x
        activations = [x]
        zs = []
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation)+b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)
        delta = (self.cost).delta(zs[-1], activations[-1], y)
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].transpose())

        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            delta = np.dot(self.weights[-l+1].transpose(), delta) * sp
            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l-1].transpose())
        return (nabla_b, nabla_w)

    def accuracy(self, data):
        results = [(np.argmax(self.feedforward(x)), y) for (x, y) in data]
        return sum(int(x == y) for (x, y) in results)

def sigmoid(z):
    return 1.0/(1.0+np.exp(-z))

def sigmoid_prime(z):
    return sigmoid(z)*(1.0-sigmoid(z))
"""

with open('network2.py', 'w') as f:
    f.write(network2_content)

print("SUCCESS: network2.py file created successfully!")

In [ ]:
import sys
import importlib
import base64
import io
import numpy as np
import matplotlib.pyplot as plt
from google.colab import output
from IPython.display import HTML, display

# Force loading the newly created network2 module
import network2
importlib.reload(network2)

# =====================================================================
# 1. MODEL INITIALIZATION AND TRAINING (ADVANCED NETWORK2)
# =====================================================================
print("="*60)
print("INITIALIZING ADVANCED NEURAL NETWORK (network2.py)...")
print("="*60)

# Architecture: 784 input neurons, 100 hidden neurons, 10 output neurons
net2 = network2.Network([784, 100, 10], cost=network2.CrossEntropyCost)

# Hyperparameters: 30 epochs, mini-batch size of 10, learning rate (eta) = 0.5, L2 regularization (lmbda) = 5.0
print("\nStarting model training over 30 epochs...")
net2.SGD(training_data, 30, 10, 0.5,
         lmbda=5.0,
         evaluation_data=test_data,
         monitor_evaluation_accuracy=True)

print("\n" + "="*60)
print("TRAINING COMPLETE! TARGET MODEL IS NOW OPTIMIZED.")
print("="*60 + "\n")

# =====================================================================
# 2. INTERACTIVE CANVAS FOR LIVE HANDWRITTEN DIGIT INFERENCE
# =====================================================================
canvas_html = """
<canvas width="280" height="280" style="border:5px solid #4CAF50; background-color:black; cursor:crosshair;"></canvas>
<br><br>
<button id="predict_btn" style="padding:12px 24px; background-color:#4CAF50; color:white; font-size:16px; border:none; border-radius:4px; cursor:pointer; font-weight:bold;">Predict Digit!</button>
<script>
var canvas = document.querySelector('canvas');
var ctx = canvas.getContext('2d');
ctx.strokeStyle = 'white';
ctx.lineWidth = 24;  // Emulates MNIST line thickness
ctx.lineCap = 'round';
ctx.lineJoin = 'round';

var drawing = false;

canvas.addEventListener('mousedown', function(e) { drawing = true; ctx.beginPath(); ctx.moveTo(e.offsetX, e.offsetY); });
canvas.addEventListener('mousemove', function(e) { if (drawing) { ctx.lineTo(e.offsetX, e.offsetY); ctx.stroke(); } });
canvas.addEventListener('mouseup', function() { drawing = false; });
canvas.addEventListener('mouseleave', function() { drawing = false; });

var button = document.querySelector('#predict_btn');
var p = new Promise(function(resolve, reject) {
    button.addEventListener('click', function() {
        resolve(canvas.toDataURL('image/png'));
    });
});
</script>
"""

print("Draw a digit (0-9) inside the black box, then click the green button:\n")
display(HTML(canvas_html))

# Capture drawing data via JavaScript promise injection
img_data = output.eval_js('p')
metadata, base64_data = img_data.split(',')
img_bytes = base64.b64decode(base64_data)

# =====================================================================
# 3. ADVANCED IMAGE PROCESSING (PADDING & BOUNDING BOX CENTERING)
# =====================================================================
from PIL import Image

image = Image.open(io.BytesIO(img_bytes)).convert('L')

bbox = image.getbbox()
if bbox:
    cropped_image = image.crop(bbox)
    max_dim = max(cropped_image.size)
    padding = int(max_dim * 0.25)

    square_size = max_dim + 2 * padding
    padded_image = Image.new('L', (square_size, square_size), 0)
    padded_image.paste(cropped_image, ((square_size - cropped_image.size[0]) // 2,
                                       (square_size - cropped_image.size[1]) // 2))

    image = padded_image.resize((28, 28), Image.Resampling.LANCZOS)
else:
    image = image.resize((28, 28), Image.Resampling.LANCZOS)

img_array = np.array(image) / 255.0
input_vector = np.reshape(img_array, (784, 1))

# =====================================================================
# 4. FORWARD PASS EXECUTION & PREDICTION OUTPUT
# =====================================================================
predictions = net2.feedforward(input_vector)
predicted_digit = np.argmax(predictions)

print("\n" + "="*50)
print(f"   Neural Network Prediction: {predicted_digit}   ")
print("="*50 + "\n")

plt.figure(figsize=(3, 3))
plt.imshow(img_array, cmap='gray')
plt.title("Processed Input Viewport (28x28)", fontsize=10)
plt.axis('off')
plt.show()